In [1]:
import plotly
import numpy as np
import plotly.graph_objs as go

In [2]:
# consider a fully actuated sheet (i.e., all the joints are controllable)
import pinocchio as pin
SDF_PATH="/home/ds3a/dev/wadiyan_carpet/models/mesh/mass_mesh_open_tree_el.sdf"
# model, constraints = pin.buildModelFromSdf("mass_mesh_open_tree.sdf", root_link_name="mass_x0_y0")
model, constraints = pin.buildModelFromSdf(SDF_PATH, root_link_name="mass_x0_y0")
# model = pin.buildModelFromSdf("spring_dampers_open_tree.sdf", root_link_name="mass_x0_y0")
# model = pin.buildModelFromSdf("mass_mesh_open_tree.sdf")
data = model.createData()

In [3]:
len(model.frames)

242

In [4]:
# Store all the frame IDs for a grid of masses in a dictionary
# stores the IDs for masses named mass_x{i}_y{j}

frame_ids = dict()

num_elements_x = 11
num_elements_y = 11


for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
        frame_ids[(i, j)] = model.getFrameId(f"mass_x{i}_y{j}")

In [5]:
q = pin.neutral(model)
# q = pin.randomConfiguration(model)
pin.forwardKinematics(model, data, q)
link_positions = []
for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
        pin.updateFramePlacement(model, data, frame_ids[(i, j)])
        # print(f"Frame mass_x{i}_y{j} placement:\n{data.oMf[frame_ids[(i, j)]].translation}\n")
        link_positions.append(data.oMf[frame_ids[(i, j)]].translation)

In [6]:
side_length = 0.4
u_min, u_max = -side_length/2, side_length/2
v_min, v_max = -side_length/2, side_length/2

# Resolution of the grid (higher = smoother)
num_u = 50
num_v = 50

u = np.linspace(u_min, u_max, num_u)
v = np.linspace(v_min, v_max, num_v)

goals_u = np.linspace(u_min, u_max, num_elements_x)
goals_v = np.linspace(v_min, v_max, num_elements_y)

U, V = np.meshgrid(u, v)
goals_U, goals_V = np.meshgrid(goals_u, goals_v)

A = 0.02
angle = np.pi/4
phase = np.pi/2
frequency = np.pi/0.4


# 2. Define your gamma(u, v)
def gamma_sur(u, v, A=0.5, base_pos=np.array([0, 0, 0])):
    x = u
    y = v
    s = u * np.cos(angle) + v * np.sin(angle)

    # z = A * np.sin(0.7*v + np.pi/6) * np.cos(u) # bump
    z = A * np.cos(frequency*s + phase) # bump

    # TODO modify x, y, and z such that (0, 0) lies at the base_pos
    offset = base_pos - np.array([0, 0, A * np.cos(frequency*(0) + phase)])
    #                                                          s = 0 at (0,0)
    x += offset[0]
    y += offset[1]
    z += offset[2]

    return x, y, z


goal_positions = dict()



X, Y, Z = gamma_sur(U, V, A=A)
goals_X, goals_Y, goals_Z = gamma_sur(goals_U, goals_V, A=A)

# 3. Make the Plotly surface figure
fig = go.Figure(data=[
    go.Surface(x=X, y=Y, z=Z),
    # go.Scatter3d(x=goals_X.flatten(), y=goals_Y.flatten(), z=goals_Z.flatten(), 
            # mode='markers', marker=dict(size=23, color='red')),
    go.Scatter3d(x=[pos[0] for pos in link_positions],y=[pos[1] for pos in link_positions],z=[pos[2] for pos in link_positions],
            mode='markers', marker=dict(size=12, color='blue')),
])

fig.update_layout(
    title="γ(u, v): bump surface",
    scene=dict(
        aspectmode='data',
        xaxis_title="u",
        yaxis_title="v",
        zaxis_title="height"
    )
)

# 4. Show the interactive plotmass_mesh_open_tree
# fig.show()

plotly.io.write_html(fig, file="bump_surface.html", auto_open=True)


In [7]:
# constraints need to be formulated from WE and EW
# we will create a list of frames that need to be linked via constraints
constraint_frames = []
for i in range(-int(num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(-int(num_elements_y/2), int(num_elements_y/2)):
        if i != 0:
            constraint_frames.append([(i, j), (i, j+1)])

len(constraint_frames)

# number of constraints = (n-1)^2

100

In [8]:
elem_dist = side_length / (num_elements_x - 1)

def get_cJacobian(model, data, q, frame_id1, frame_id2, dist):
    pin.computeJointJacobians(model, data, q)
    pin.updateFramePlacements(model, data)

    pin.forwardKinematics(model, data, q)

    J1 = pin.getFrameJacobian(model, data, frame_id1, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
    J2 = pin.getFrameJacobian(model, data, frame_id2, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)

    p1 = data.oMf[frame_id1].translation
    p2 = data.oMf[frame_id2].translation

    constraint_fn_der_wrt_FK = 2*(p1 - p2).reshape(1, 3)
    # Constraint: c(q) = ||p1(q) - p2(q)||^2
    # dc/dq = 2*(p1 - p2)^T * (J2 - J1)
    # (sign is arbitrary for constraints)

    # relative Jacobian
    J_rel = J2 - J1
    J_rel = J_rel[0:3, :]  # only position constraints
    # print(J_rel.shape, constraint_fn_der_wrt_FK.shape)
    J_const = constraint_fn_der_wrt_FK.dot(J_rel)
    return J_const

def get_cJacobian_for_constraint(model, data, q, constraint_pair, dist):
    frame_id1 = frame_ids[constraint_pair[0]]
    frame_id2 = frame_ids[constraint_pair[1]]
    return get_cJacobian(model, data, q, frame_id1, frame_id2, dist)

def get_cJacobian_matrix(model, data, q, constraint_frames, dist):
    J_c_list = []
    for constraint_pair in constraint_frames:
        J_c = get_cJacobian_for_constraint(model, data, q, constraint_pair, dist)
        J_c_list.append(J_c)
        # print(np.linalg.det(J_c.dot(J_c.T)))
        # must be non zero for a non-singular constraint
    J_c_matrix = np.vstack(J_c_list)
    return J_c_matrix

In [9]:
Jc = get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist)
Jc.shape # should be ((n-1)^2, model.nv)

(100, 360)

In [10]:
goal_positions = dict()
# goal_positions_other = dict()
for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
        x, y, z = gamma_sur(i*elem_dist, j*elem_dist, A=A, base_pos=np.array([0, 0, 0]))
        goal_positions[(i, j)] = np.array([x, y, z])
        # goal_positions_other[(i, j)] = np.array([goals_X[j + int(num_elements_y/2), i + int(num_elements_x/2)],
        #                                    goals_Y[j + int(num_elements_y/2), i + int(num_elements_x/2)],
        #                                    goals_Z[j + int(num_elements_y/2), i + int(num_elements_x/2)]])

In [11]:
# Now, to come up with task jacobians

def get_task_jacobians(model, data, q, goal_positions, frame_ids):
    pin.computeJointJacobians(model, data, q)
    pin.updateFramePlacements(model, data)

    pin.forwardKinematics(model, data, q)

    J_t_list = []
    for frame_id in frame_ids:
        # print(frame_ids[frame_id])
        # TODO get the position of the frame from the pinocchio model using q
        # Then get the goal for that frame using the index and goal_positions
        # Finally, compute the task Jacobian for that frame using the position error
        frame_pos = data.oMf[frame_ids[frame_id]].translation
        goal_pos = goal_positions[frame_id]
        err = goal_pos - frame_pos
        # print(err)


        J = pin.getFrameJacobian(model, data, frame_ids[frame_id], pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
        J = -J[0:3, :]  # only position
        # print(np.linalg.det(J.dot(J.T)))
        J_t_list.append(J)
    J_t_matrix = np.vstack(J_t_list)
    return J_t_matrix

def get_task_jacobians_w_errors(model, data, q, goal_positions, frame_ids):
    pin.computeJointJacobians(model, data, q)
    pin.updateFramePlacements(model, data)

    pin.forwardKinematics(model, data, q)

    J_t_list = []
    errors = []
    for frame_id in frame_ids:
        # print(frame_ids[frame_id])
        # TODO get the position of the frame from the pinocchio model using q
        # Then get the goal for that frame using the index and goal_positions
        # Finally, compute the task Jacobian for that frame using the position error
        frame_pos = data.oMf[frame_ids[frame_id]].translation
        goal_pos = goal_positions[frame_id]
        err = goal_pos - frame_pos
        # print(err)


        J = pin.getFrameJacobian(model, data, frame_ids[frame_id], pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
        J = -J[0:3, :]  # only position
        # print(np.linalg.det(J.dot(J.T)))
        J_t_list.append(J)
        errors.append(err)
        # print(err.shape)
    error_vec = np.hstack(errors)
    # print(error_vec.shape)
    J_t_matrix = np.vstack(J_t_list)
    return J_t_matrix, error_vec


Opening in existing browser session.


In [12]:
Jt, errors = get_task_jacobians_w_errors(model, data, q, goal_positions, frame_ids)

Jt.shape # should be (n^2 * 3, model.nv)

(363, 360)

In [13]:
# now to make a kkt matrix

def make_kkt_matrix(Jc, Jt, reg=1e-6):
    if Jc.shape[1] != Jt.shape[1]:
        raise ValueError("Jc and Jt must have the same number of columns (variables)")
    num_vars = Jc.shape[1]
    num_constraints = Jc.shape[0]
    num_tasks = Jt.shape[0]

    G = Jt.T.dot(Jt) + reg * np.eye(num_vars)
    H = Jc

    KKT_left = np.vstack([G, H])
    KKT_right = np.vstack([H.T, np.zeros((num_constraints, num_constraints))])

    KKT_matrix = np.hstack([KKT_left, KKT_right])
    return KKT_matrix

In [14]:
KKT = make_kkt_matrix(get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist), get_task_jacobians(model, data, q, goal_positions, frame_ids))

KKT.shape

(460, 460)

In [15]:
# make the IK solver
def solve_ik_step(model, data, q, constraint_frames, goal_positions, frame_ids, reg=1e-6):
    Jc = get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist)
    Jt, errors = get_task_jacobians_w_errors(model, data, q, goal_positions, frame_ids)

    KKT = make_kkt_matrix(Jc, Jt, reg=reg)

    num_vars = Jc.shape[1]
    num_constraints = Jc.shape[0]
    num_tasks = Jt.shape[0]

    # right hand side
    b_top = -Jt.T.dot(errors)
    b_bottom = np.zeros((num_constraints, ))
    b = np.hstack([b_top, b_bottom])

    # solve for delta_q and lambda
    sol = np.linalg.solve(KKT, b)
    delta_q = sol[0:num_vars]
    # lambda_vals = sol[num_vars:]

    return delta_q

In [16]:
for i in range(5):
    v = solve_ik_step(model, data, q, constraint_frames, goal_positions, frame_ids)
    q = pin.integrate(model, q, v)



In [17]:
def run_constrained_ik(
    model,
    data,
    gamma_fn,
    side_length,
    num_elements_x,
    num_elements_y,
    A=0.05,
    base_pos=np.array([0, 0, 0]),
    num_iters=6,
    reg=1e-6,
):
    """Run the full constrained IK pipeline and return the converged configuration."""

    # 1) Build frame map for the mass grid
    frame_ids = {}
    for i in range(int(-num_elements_x / 2), int(num_elements_x / 2) + 1):
        for j in range(int(-num_elements_y / 2), int(num_elements_y / 2) + 1):
            frame_ids[(i, j)] = model.getFrameId(f"mass_x{i}_y{j}")

    # 2) Build pairwise distance constraints (same structure as above)
    constraint_frames = []
    for i in range(-int(num_elements_x / 2), int(num_elements_x / 2) + 1):
        for j in range(-int(num_elements_y / 2), int(num_elements_y / 2)):
            if i != 0:
                constraint_frames.append([(i, j), (i, j + 1)])

    # 3) Create goal positions from gamma
    elem_dist = side_length / (num_elements_x - 1)
    goal_positions_ = {}
    for i in range(int(-num_elements_x / 2), int(num_elements_x / 2) + 1):
        for j in range(int(-num_elements_y / 2), int(num_elements_y / 2) + 1):
            x, y, z = gamma_fn(i * elem_dist, j * elem_dist, A=A, base_pos=base_pos)
            goal_positions_[(i, j)] = np.array([x, y, z])
    # print("Goal positions:\n", goal_positions_)

    # 4) Initialize at neutral and run constrained IK iterations
    q = pin.neutral(model)
    err = None
    for _ in range(num_iters):
        Jc = get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist)
        Jt, errors = get_task_jacobians_w_errors(model, data, q, goal_positions_, frame_ids)

        KKT = make_kkt_matrix(Jc, Jt, reg=reg)
        num_vars = Jc.shape[1]
        num_constraints = Jc.shape[0]

        b_top = -Jt.T.dot(errors)
        b_bottom = np.zeros((num_constraints,))
        b = np.hstack([b_top, b_bottom])

        sol = np.linalg.solve(KKT, b)
        delta_q = sol[0:num_vars]
        q = pin.integrate(model, q, delta_q)
        err = errors
        # print(np.sum(errors**2)/len(errors))

    return q, frame_ids, constraint_frames, goal_positions_, err


q, frame_ids, constraint_frames, goal_positions, err = run_constrained_ik(
    model,
    data,
    gamma_sur,
    side_length=side_length,
    num_elements_x=num_elements_x,
    num_elements_y=num_elements_y,
    A=A,
    base_pos=np.array([0, 0, 0]),
    num_iters=5,
)

link_positions = []
for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
        pin.updateFramePlacement(model, data, frame_ids[(i, j)])
        # print(f"Frame mass_x{i}_y{j} placement:\n{data.oMf[frame_ids[(i, j)]].translation}\n")
        link_positions.append(data.oMf[frame_ids[(i, j)]].translation)


In [18]:
import plotly.io as pio


import numpy as np

# Parameters you can tune
distance = 3.0        # how far from center
yaw = np.deg2rad(75)  # rotation around z
pitch = np.deg2rad(10)  # tilt up/down

# Convert spherical → Cartesian
eye_x = distance * np.cos(pitch) * np.cos(yaw)
eye_y = distance * np.cos(pitch) * np.sin(yaw)
eye_z = distance * np.sin(pitch)

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# ---------- your grid ----------
side_length = 0.4
u_min, u_max = -side_length/2, side_length/2
v_min, v_max = -side_length/2, side_length/2
num_u = 60
num_v = 60
u = np.linspace(u_min, u_max, num_u)
v = np.linspace(v_min, v_max, num_v)
U, V = np.meshgrid(u, v)

# ---------- surface params ----------
A = 0.03
frequency = np.pi/0.4

# choose your sweep
angles = np.deg2rad([0, 45])
phases = [0.0, np.pi/4, np.pi]

rows, cols = len(angles), len(phases)

# ---------- precompute z-range for consistent geometry scale ----------
Z_all = []
for ph in phases:
    for ang in angles:
        angle = ang
        phase = ph
        _, _, Z = gamma_sur(U, V, A=A, base_pos=np.array([0, 0, 0]))
        Z_all.append(Z)
zmin = min(Z.min() for Z in Z_all)
zmax = max(Z.max() for Z in Z_all)

# ---------- camera (shared) ----------
camera = dict(
    eye=dict(x=eye_x, y=eye_y, z=eye_z),
    center=dict(x=0.0, y=0.0, z=0.0),
    up=dict(x=0.0, y=0.0, z=1.0),
)

# ---------- make subplot figure (3D scenes) ----------
fig = make_subplots(
    rows=rows, cols=cols,
    specs=[[{"type": "scene"} for _ in range(cols)] for _ in range(rows)],
    subplot_titles=[
        f"angle={np.rad2deg(ang):.0f}°, phase={np.rad2deg(ph):.0f}°"
        for ang in angles for ph in phases
    ],
    horizontal_spacing=0.02,
    vertical_spacing=0.1,
)

for r, ang in enumerate(angles, start=1):
    for c, ph in enumerate(phases, start=1):
        angle = ang
        phase = ph

        # special-case frequency requested by user
        # frequency = np.pi / 0.8 if np.isclose(angle, np.pi / 4) and np.isclose(phase, np.pi / 4) else np.pi / 0.4

        X, Y, Z = gamma_sur(U, V, A=A, base_pos=np.array([0, 0, 0]))

        # perform constrained IK for this angle/phase and get link positions
        q_local, constraint_frames, frame_ids_local, goals_, err  = run_constrained_ik(
            model,
            data,
            gamma_sur,
            side_length=side_length,
            num_elements_x=num_elements_x,
            num_elements_y=num_elements_y,
            A=A,
            base_pos=np.array([0, 0, 0]),
            num_iters=1,
        )
        print(len(err))

        mse = np.sum(100*err**2)/(len(err)/3) # convert to mm and average over all points

        title_index = (r - 1) * cols + (c - 1)

        fig.layout.annotations[title_index].text = (
            f"Angle={np.rad2deg(ang):.0f}°, "
            f"Phase={np.rad2deg(ph):.0f}°, "
            f"MSE={mse:.4e} mm"
        )

        pin.forwardKinematics(model, data, q_local)
        pin.updateFramePlacements(model, data)
        

        link_positions_ = []
        goal_positions = dict()
        for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
            for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
                x, y, z = gamma_sur(i*elem_dist, j*elem_dist, A=A, base_pos=np.array([0, 0, 0]))
                goal_positions[(i, j)] = np.array([x, y, z])
                # fill link_positions_ in the same order as the frames were indexed
                link_pos = data.oMf[frame_ids[(i, j)]].translation
                link_positions_.append(link_pos)

        fig.add_trace(
            go.Surface(
                x=X, y=Y, z=Z,
                coloraxis="coloraxis",
                showscale=False,
            ),
            row=r, col=c
        )

        show_dots_legend = (r == 1 and c == 1)  # only first subplot gets legend entry

        fig.add_trace(
            go.Scatter3d(
                x=[i[0] for i in link_positions_],
                y=[i[1] for i in link_positions_],
                z=[i[2] for i in link_positions_],
                mode='markers',
                marker=dict(size=4, color='cyan', line=dict(color="black", width=1)),
                showlegend=show_dots_legend,
                name="Mass elements",  # <-- the one label you want
                legendgroup="points"
            ),
            row=r, col=c
        )

# Single shared colorbar + fixed color scale range
fig.update_layout(
    coloraxis=dict(
        colorscale="inferno",
        cmin=zmin, cmax=zmax,
        colorbar=dict(title="z (m)", len=0.85)
    ),
    width=2000,
    height=900,
    margin=dict(l=10, r=10, t=80, b=10),
)

fig.update_layout(
    legend=dict(
            # x=0.1,
            # y=0.55,
            xanchor="right",
            yanchor="top", 
        font=dict(
            family="Times New Roman",
            size=24,
        ),
        itemsizing="constant",
    )
)

for annotation in fig.layout.annotations:
    annotation.font.size = 24
    annotation.font.family = "Times New Roman"

# Apply same camera + axis ranges to every scene
for i in range(1, rows*cols + 1):
    scene_key = "scene" if i == 1 else f"scene{i}"
    fig.layout[scene_key].update(
        camera=camera,
        xaxis=dict(title="u (m)"),
        yaxis=dict(title="v (m)"),
        zaxis=dict(title="z (m)", nticks=7),
        aspectmode="manual",
        aspectratio=dict(x=3, y=3, z=0.75),
    )

# Export one PNG
pio.write_image(fig, "grid_angle_phase.png", scale=4)
# Or interactive HTML:
pio.write_html(fig, "grid_angle_phase.html", auto_open=True)


363
363
363
363
363
363


Opening in existing browser session.
